# 1. Instalando as Bibliotecas

In [21]:
!pip -q install \
langchain \
langchain-community \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

# 2. Importando as bibliotecas

In [22]:
import os

from google.colab import userdata

# Documentos
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Divisão de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Banco Vetorial
from langchain_community.vectorstores import FAISS

# Modelo Groq
from langchain_groq import ChatGroq

# Prompt
from langchain_core.prompts import ChatPromptTemplate

# Chains
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("✅ Bibliotecas carregadas.")

✅ Bibliotecas carregadas.


# 3. Clonar o repositório do GitHub

In [23]:
!rm -rf PortfolioAI

!git clone https://github.com/elissouza2023/PortfolioAI.git

BASE_PATH = "/content/PortfolioAI"

KNOWLEDGE_PATH = f"{BASE_PATH}/knowledge_base"

VECTOR_PATH = f"{BASE_PATH}/vector_store"

os.makedirs(VECTOR_PATH, exist_ok=True)

print("✅ Repositório clonado.")

Cloning into 'PortfolioAI'...
remote: Enumerating objects: 234, done.
remote: Total 234 (delta 0), reused 0 (delta 0), pack-reused 234 (from 1)
Receiving objects: 100% (234/234), 37.61 MiB | 24.04 MiB/s, done.
Resolving deltas: 100% (102/102), done.
✅ Repositório clonado.


# 4. Configurar API Key da Groq

In [24]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API Key carregada.")

✅ API Key carregada.


# 5. Carregando documentos da pasta knowledge_base

In [25]:
loader = PyPDFDirectoryLoader(KNOWLEDGE_PATH)

documents = loader.load()

print(f"\n📄 Total de documentos: {len(documents)}")

for doc in documents:
    print(doc.metadata["source"])


📄 Total de documentos: 83
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Curriculo_Elisangela_de_Souza resumido TI.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Currículo geral.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Perfil Profissional – Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/content/PortfolioAI/knowledge_base/Competências Técnicas - Elisângela de Souza.pdf
/co

# 6. Dividindo os documentos em chunks

In [26]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=900,

    chunk_overlap=150,

    separators=[
        "\n\n",
        "\n",
        ".",
        "!",
        "?",
        " "
    ]
)

texts = text_splitter.split_documents(documents)

print(f"✅ Chunks criados: {len(texts)}")

✅ Chunks criados: 186


# 7. Criando Embeddings

In [27]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 8. Banco Vetorial

In [28]:
vector_store = FAISS.from_documents(

    texts,

    embeddings

)

vector_store.save_local(VECTOR_PATH)

print("✅ Banco vetorial criado.")

✅ Banco vetorial criado.


In [29]:
from google.colab import files
import os

# Compacta a pasta
!zip -r vector_store.zip /content/PortfolioAI/vector_store

# Faz o download
files.download("vector_store.zip")

updating: content/PortfolioAI/vector_store/ (stored 0%)
updating: content/PortfolioAI/vector_store/index.faiss (deflated 7%)
updating: content/PortfolioAI/vector_store/index.pkl (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 9. Modelo Groq

In [50]:
MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(

    model_name=MODEL_NAME,

    temperature=0.2,

    max_tokens=1200

)

print("✅ Modelo carregado.")

✅ Modelo carregado.


## 10. Prompt do PortfolioAI

In [51]:
system_prompt = """
Você é o PortfolioAI.

Seu objetivo é responder perguntas sobre Elisângela de Souza.

REGRAS IMPORTANTES

• Utilize EXCLUSIVAMENTE as informações presentes no contexto.

• Nunca invente experiências.

• Nunca complete informações por conta própria.

• Caso não exista resposta no contexto, diga:

"Não encontrei essa informação na minha base de conhecimento.
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela."

• Sempre escreva de forma profissional, porém em primeira pessoal como em uma entrevista, lembre-se de que vocÊ é o meu asistente pessoal. Por exemplo diga sou uma profissional...

• Manetenha racioninio fluído, linguagem clara, tom entusiastico. Mantenha o tom de conversa, como em uma entrevista de emprego.

• Sempre responda em português.

• Quando possível organize a resposta em tópicos.

Contexto:

{context}
"""

prompt = ChatPromptTemplate.from_messages(

    [

        ("system", system_prompt),

        ("human", "{input}")

    ]

)

print("✅ Prompt criado.")

✅ Prompt criado.


# 11. Chain RAG

In [52]:
question_answer_chain = create_stuff_documents_chain(

    llm,

    prompt

)

retriever = vector_store.as_retriever(

    search_kwargs={

        "k":6

    }

)

rag_chain = create_retrieval_chain(

    retriever,

    question_answer_chain

)

print("✅ RAG criado.")

✅ RAG criado.


# 12. Função para perguntas

In [53]:
def perguntar(pergunta):

    resposta = rag_chain.invoke(

        {

            "input": pergunta

        }

    )

    print("="*80)

    print("PERGUNTA")

    print(pergunta)

    print()

    print("RESPOSTA")

    print(resposta["answer"])

    print()

    print("FONTES UTILIZADAS")

    fontes = set()

    for doc in resposta["context"]:

        fontes.add(

            os.path.basename(

                doc.metadata["source"]

            )

        )

    for fonte in sorted(fontes):

        print("•", fonte)

    print("="*80)

# 13. Testes

In [54]:
perguntar("Quem é a Elisângela?")

PERGUNTA
Quem é a Elisângela?

RESPOSTA
**Sou Elisângela de Souza**, natural de Volta Redonda – RJ, e apresento-me como uma profissional que combina experiência operacional e administrativa na indústria com uma transição estratégica para a área de Tecnologia da Informação.

### Principais pontos sobre mim

- **Formação Acadêmica**  
  - Bacharel em Administração de Empresas.  
  - Pós‑graduação em Engenharia Metalúrgica (concluída).  
  - Pós‑graduação em Engenharia de Equipamentos (incompleta).  
  - Tecnologia em Segurança da Informação (cursando).  
  - Técnico em Administração de Empresas.  
  - Técnico em Contabilidade (curso complementar).

- **Experiência Profissional**  
  - **CSN‑UPV – Volta Redonda (RJ)** – Operadora de Produção II desde junho de 2013.  
  - Atuação em equipe, resolução de problemas antes da escolha de tecnologias e foco na compreensão profunda do problema para desenvolver soluções eficazes.

- **Objetivo Profissional**  
  - Atuar nas áreas de Tecnologia da 

In [55]:
perguntar("Qual é o seu objetivo profissional ?")

PERGUNTA
Qual é o seu objetivo profissional ?

RESPOSTA
**Objetivo profissional**

Sou uma profissional que busca, de forma constante, transformar desafios em oportunidades por meio da tecnologia e da Inteligência Artificial. Meu objetivo principal é:

- **Desenvolver soluções inteligentes** que sejam centradas nas pessoas, atendendo às necessidades reais dos usuários e gerando valor sustentável para as organizações.  
- **Aplicar inovação** em processos e produtos, integrando conhecimentos de IA, análise de dados e boas práticas de gestão de projetos.  
- **Promover um ambiente colaborativo**, onde o feedback e o aprendizado contínuo são ferramentas essenciais para o crescimento individual e coletivo.  
- **Contribuir para o sucesso dos projetos** alinhando minhas competências técnicas e comportamentais (responsabilidade, disciplina, organização, colaboração e adaptação) aos objetivos estratégicos da empresa.  

Em resumo, meu objetivo profissional é criar e implementar soluções tecno

In [56]:
perguntar("Quais projetos ela desenvolveu?")

PERGUNTA
Quais projetos ela desenvolveu?

RESPOSTA
Não encontrei essa informação na minha base de conhecimento.  
Caso deseje mais detalhes, recomendo entrar em contato diretamente com a Elisângela.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Projetos e Cases – Elisângela de Souza.pdf
• Trajetória Profissional – Elisângela de Souza.pdf


In [57]:
perguntar("Fale sobre o projeto PortfolioAI.")

PERGUNTA
Fale sobre o projeto PortfolioAI.

RESPOSTA
**Sobre o projeto PortfolioAI**

Sou uma profissional que concebeu o **PortfolioAI** como uma base de conhecimento estruturada para organizar, apresentar e disponibilizar de forma clara e escalável os principais projetos e cases que desenvolvi ao longo da minha trajetória de formação e transição para a área de Tecnologia da Informação.

### Principais objetivos
- **Centralizar informações** sobre meus projetos, competências e processos de aprendizagem contínua.
- **Facilitar respostas** a perguntas recorrentes, como:
  - Quais projetos eu desenvolvi?
  - Tenho experiência em Machine Learning, UX ou Segurança da Informação?
  - Qual a minha experiência com Python?
  - Como costumo estruturar meus projetos?
- **Demonstrar a evolução** da minha carreira, destacando a adaptação a diferentes contextos (administração, processos industriais, TI) e a busca constante por atualização.

### Estrutura e organização
- **Divisão em três grupos** d

In [58]:
perguntar("Quais competências técnicas ela possui?")

PERGUNTA
Quais competências técnicas ela possui?

RESPOSTA
Não encontrei essa informação na minha base de conhecimento.  
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela.

FONTES UTILIZADAS
• Competências Comportamentais – Elisângela de Souza.pdf
• Trajetória Profissional – Elisângela de Souza.pdf


In [59]:
perguntar("Qual sua formação acadêmica?")

PERGUNTA
Qual sua formação acadêmica?

RESPOSTA
Não encontrei essa informação na minha base de conhecimento.  
Caso deseje mais detalhes, recomendo entrar em contato diretamente com Elisângela.

FONTES UTILIZADAS
• Formação Acadêmica - Elisângela de Souza.pdf
